# Create Wealth Maps

Builds a sampling grid over an area of interest (a country's ADM0 boundary by default, or an arbitrary polygon), extracts imagery/covariate/embedding features at grid points directly from the per-country COG mosaics produced by `notebook/preprocessing/imagery/export_country_composites.ipynb`, runs the trained CNN and/or XGBoost models, and assembles the resulting predictions into GeoTIFF wealth map(s).

A single `OUTPUT_MAP_RESOLUTION` drives one shared sampling grid used by both products:
- **CNN map**: samples Sentinel/Landsat patches at each grid point and runs the trained ResNet50 models (5-fold ensemble). `OUTPUT_MAP_RESOLUTION` must exactly equal the CNN patch footprint (`PATCH_SIZE * IMAGERY_SCALE[...]`), so each grid cell is exactly one CNN patch.
- **XGBoost map**: samples spatial covariates and/or AlphaEarth embeddings at the same grid -- a single pixel when `OUTPUT_MAP_RESOLUTION` matches a source's own native resolution, or a nodata-masked spatial mean over the point's footprint otherwise (a "zonal mean", computed via `PointGeoSampler`'s CRS-unit windowing rather than a separate zonal-stats step). Because both branches share one grid, XGBoost inference can freely combine spatial covariates/AEF with CNN embeddings (e.g. `resnet-spatial`) whenever both branches are run.

**Prerequisite**: the XGBoost branch needs a Covariates and/or AlphaEarth COG for `MAP_COUNTRY`; the CNN branch additionally needs a Sentinel/Landsat COG. `export_country_composites.ipynb`'s `IMAGERY_TYPES` must include the relevant type(s) (`'SENTINEL'`/`'LANDSAT'` are not built by default) before running this notebook.

In [ ]:
import math
import torch
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
import xgboost as xgb
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from torchgeo.samplers import Units
from torchgeo.datasets.utils import stack_samples
from tqdm import tqdm

from maw.utils import build_grid, grid_transform, get_utm_epsg, reduce_spatial_mean
from maw.utils.samplers import PointGeoSampler
from maw.datasets import SpatialCovariates, AlphaEarth, Sentinel, Landsat, EXPECTED_COVARIATES
from maw.datasets.sentinel import to_satlas_band_order, SATLAS_MIN_INT, SATLAS_MAX_INT
from maw.preprocessing import ClampAndRescale, DequantiseAEF
from maw.models.resnet import build_resnet50

## Config

Everything needed to reproduce a specific wealth map -- change these, then run the whole notebook top to bottom.

In [ ]:
# --- Area of Interest
MAP_COUNTRY = "PHL"      # ISO3 country code; keys default boundary + COG lookups
MAP_YEAR = 2020          # year of imagery to use; must be available in data-raw/imagery/country/cogs
BOUNDARY_PATH = None     # optional override polygon path; defaults to data-raw/boundaries/ADM0/{country}_adm0.geojson when None

In [ ]:
# --- Paths
HOME_DIR = Path.home() / "Projects/mapping-asia-wealth/"
BOUNDARIES_DIR = HOME_DIR / "data-raw/boundaries/"             # dir with ADM0 country boundaries
IMAGERY_ROOT = HOME_DIR / "data-raw/imagery/country/cogs/"     # expected format {country}_{sentinel|landsat|covs|aef}_s{year}.tif

MAPS_DIR = HOME_DIR / "data-clean/outputs/maps/"
GRID_DIR = MAPS_DIR / "map-grids/"
CNN_PREDICTIONS_OUTDIR = MAPS_DIR / "map-cnn-predictions/"
XGB_PREDICTIONS_OUTDIR = MAPS_DIR / "map-xgb-predictions/"
RESNET_EMBEDS_DIR = HOME_DIR / "data-clean/outputs/embeddings/sentinel-resnet/"
TABULAR_FEATURES_DIR = HOME_DIR / "data-clean/outputs/features/country-cogs/"

CHECKPOINT_DIR = HOME_DIR / "data-clean/models/SENTINEL-RESNET50"       # CNN checkpoints
XGB_MODEL_DIR = HOME_DIR / "data-clean/models/xgboost/"                # trained XGBoost models

for d in (GRID_DIR, CNN_PREDICTIONS_OUTDIR, XGB_PREDICTIONS_OUTDIR, RESNET_EMBEDS_DIR, TABULAR_FEATURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Params

# Product(s) to build this run
BUILD_CNN_MAP = False
BUILD_XGB_MAP = True

# Output resolution: sets the shared sampling grid's spacing, and (compared against each
# source's own native resolution) whether spatial covariates/AEF embeddings are point-sampled
# or window-averaged via the torchgeo sampler -- see the tabular extraction cell below.
OUTPUT_MAP_RESOLUTION = 500

# Input Params
PATCH_SIZE = 512                                              # CNN input patch size, in pixels
IMAGERY_SCALE = {"SENTINEL": 10, "LANDSAT": 30}               # native COG resolution (m/px) per imagery (must match input imagery scale)
CNN_IMAGERY_TYPE = "SENTINEL"                                 # which COG the CNN branch samples patches from

if BUILD_CNN_MAP:
    # Each CNN patch must cover exactly one output-map pixel, so any Sentinel/Landsat
    # embeddings extracted below also cover the correct area.
    expected_resolution = PATCH_SIZE * IMAGERY_SCALE[CNN_IMAGERY_TYPE]
    assert OUTPUT_MAP_RESOLUTION == expected_resolution, (
        f"OUTPUT_MAP_RESOLUTION ({OUTPUT_MAP_RESOLUTION}m) must equal PATCH_SIZE * IMAGERY_SCALE[{CNN_IMAGERY_TYPE!r}] "
        f"({PATCH_SIZE} * {IMAGERY_SCALE[CNN_IMAGERY_TYPE]} = {expected_resolution}m). Adjust PATCH_SIZE "
        "(or OUTPUT_MAP_RESOLUTION) so the CNN patch footprint matches the output grid."
    )

# Loader Params
BATCH_SIZE = 64
LOADER_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CNN Model Params
FC_HIDDEN = 1024
INITIAL_WEIGHTS = "SENTINEL2_SI_MS_SATLAS"
CKPT_STEM = "RESNET50-NOINDIAWEIGHTS-LR0.001-WD0.001-DP0.3"   # stem of trained CNN checkpoint names

In [ ]:
# --- XGBoost Feature Configs
TABULAR_BATCH_SIZE = 256
TABULAR_LOADER_WORKERS = 4

# One canonical feature-type table. Each entry only runs if all of its `requires` feature
# sources are available this run ("cog_covariates"/"cog_aef" need the matching COG;
# "resnet_embeddings" needs BUILD_CNN_MAP=True). To reproduce a higher-resolution
# XGBoost-only map instead, swap in the "XGB-HIGHRES-*" model stems here and set
# OUTPUT_MAP_RESOLUTION to the covariates/AEF COGs' own native resolution -- dropped from
# this default config for the replication package.
XGB_FEATURE_CONFIGS = [
    {"feature_type": "spatial-only",   "model_stem": "SENTINEL-XGBOOST-BESTPARAMS-SPATIAL-ONLY",    "requires": ["cog_covariates"]},
    {"feature_type": "resnet-only",    "model_stem": "SENTINEL-XGBOOST-BESTPARAMS-EMBEDDINGS-ONLY", "requires": ["resnet_embeddings"]},
    {"feature_type": "resnet-spatial", "model_stem": "SENTINEL-XGBOOST-BESTPARAMS-ALL-FEATURES",    "requires": ["cog_covariates", "resnet_embeddings"]},
    {"feature_type": "aef-only",       "model_stem": "AEF-XGBOOST-EMBEDDINGS-ONLY-BESTPARAMS",      "requires": ["cog_aef"]},
    {"feature_type": "aef-spatial",    "model_stem": "AEF-XGBOOST-ALL-FEATURES-BESTPARAMS",         "requires": ["cog_covariates", "cog_aef"]},
]

## Helpers

Small notebook-local helpers for locating a country's COGs and writing the final GeoTIFF -- kept here rather than in `maw` since they're thin, notebook-specific glue around the reusable pieces in `maw.utils`.

In [ ]:
def find_country_cog(imagery_root, country, imagery_type, imagery_year):
    """Return the path to a country's COG for the given imagery type, or None if missing."""
    matches = sorted(Path(imagery_root).glob(f"{country.lower()}_{imagery_type.lower()}_s{imagery_year}*.tif"))
    if len(matches) == 0:
        return None
    if len(matches) > 1:
        raise ValueError(f"Expected exactly one {imagery_type} COG for {country}, found {len(matches)}: {matches}")
    return matches[0]


def write_wealth_map(full_grid, values_df, out_path, resolution_m, epsg_crs, nodata=-9999.0):
    """Assemble per-patch_id predictions into a raster over `full_grid` and export as GeoTIFF."""
    ny = int(full_grid["grid_y"].max() + 1)
    nx = int(full_grid["grid_x"].max() + 1)
    transform = grid_transform(full_grid, resolution_m)

    merged = full_grid.merge(values_df, on="patch_id", how="left").sort_values(["grid_y", "grid_x"])
    arr = merged["wealth_index_mean"].fillna(nodata).to_numpy(dtype=np.float32).reshape(ny, nx)

    plt.imshow(arr, cmap="grey")
    plt.title(out_path.stem)
    plt.show()

    with rasterio.open(
        out_path, "w", driver="GTiff", height=ny, width=nx, count=1,
        dtype=np.float32, crs=epsg_crs, transform=transform, nodata=nodata,
        compress="deflate", tiled=True, blockxsize=512, blockysize=512,
    ) as dst:
        dst.write(arr, 1)
    print(f"Wrote {out_path}")

## Load AOI Boundary

Defaults to the country's ADM0 boundary; set `BOUNDARY_PATH` above to build the grid over an arbitrary polygon instead (e.g. a coastal buffer).

In [ ]:
if BOUNDARY_PATH is not None:
    boundary = gpd.read_file(BOUNDARY_PATH)
else:
    boundary = gpd.read_file(BOUNDARIES_DIR / "ADM0" / f"{MAP_COUNTRY.lower()}_adm0.geojson")

boundary = boundary.to_crs("EPSG:4326").dissolve()
boundary_wgs84 = boundary.geometry.iloc[0]

centroid = boundary_wgs84.centroid
GRID_CRS = f"EPSG:{get_utm_epsg(centroid.x, centroid.y)}"

boundary_utm = boundary.to_crs(GRID_CRS)
boundary_polygon = boundary_utm.geometry.iloc[0]

ax = boundary_utm.plot(facecolor="none", edgecolor="black", linewidth=2)
ax.set_title(f"{MAP_COUNTRY} AOI boundary ({GRID_CRS})")
plt.show()

## Build Sampling Grid

One shared grid, at `OUTPUT_MAP_RESOLUTION`, feeds both the CNN and XGBoost branches below. It's snapped to a source COG's own pixel grid when possible -- the CNN imagery COG when `BUILD_CNN_MAP`, else the AEF/covariates COG -- so grid points land on exact pixel boundaries and the output raster aligns 1:1 with the source imagery. Otherwise it falls back to a fresh grid derived purely from the AOI polygon bounds and the requested resolution -- the path used for a brand-new custom polygon/resolution with no matching COG yet.

In [ ]:
# Locate necessary input data
covs_cog_path = find_country_cog(IMAGERY_ROOT, MAP_COUNTRY, "covs", MAP_YEAR)
aef_cog_path = find_country_cog(IMAGERY_ROOT, MAP_COUNTRY, "aef", MAP_YEAR)
cnn_cog_path = find_country_cog(IMAGERY_ROOT, MAP_COUNTRY, CNN_IMAGERY_TYPE, MAP_YEAR)

if BUILD_XGB_MAP and covs_cog_path is None and aef_cog_path is None:
    raise FileNotFoundError(
        f"BUILD_XGB_MAP=True but no covariates or AEF COG found for {MAP_COUNTRY} under {IMAGERY_ROOT}. "
        "Run export_country_composites.ipynb with IMAGERY_TYPES including 'COVS'/'AEF' first."
    )
if BUILD_CNN_MAP and cnn_cog_path is None:
    raise FileNotFoundError(
        f"BUILD_CNN_MAP=True but no {CNN_IMAGERY_TYPE} COG found for {MAP_COUNTRY} under {IMAGERY_ROOT}. "
        f"Run export_country_composites.ipynb with IMAGERY_TYPES including '{CNN_IMAGERY_TYPE}' first "
        "-- it is not included by default (see notebook/README.md)."
    )

In [ ]:
if BUILD_CNN_MAP or BUILD_XGB_MAP:

    if BUILD_CNN_MAP:
        # OUTPUT_MAP_RESOLUTION == PATCH_SIZE * IMAGERY_SCALE[...] is already asserted above,
        # so the grid always aligns exactly to the CNN imagery COG's own pixel grid.
        with rasterio.open(cnn_cog_path) as src:
            align_to = src.transform
    else:
        align_cog_path = aef_cog_path or covs_cog_path   # prefer AEF's transform when both exist
        with rasterio.open(align_cog_path) as src:
            native_res = src.transform.a
            native_transform = src.transform

        ratio = OUTPUT_MAP_RESOLUTION / native_res
        if math.isclose(ratio, round(ratio), rel_tol=1e-3) and round(ratio) >= 1:
            align_to = native_transform
            print(f"OUTPUT_MAP_RESOLUTION ({OUTPUT_MAP_RESOLUTION}m) is a multiple of the native "
                  f"COG resolution ({native_res}m) -> snapping the grid to the COG's pixel grid.")
        else:
            align_to = None
            print(f"OUTPUT_MAP_RESOLUTION ({OUTPUT_MAP_RESOLUTION}m) does not evenly divide the "
                  f"native COG resolution ({native_res}m) -> building a fresh grid; covariate/AEF "
                  "samples will be a spatial mean over each point's footprint instead of a single pixel.")

    grid_full, grid = build_grid(
        boundary_polygon, crs=GRID_CRS, resolution_m=OUTPUT_MAP_RESOLUTION,
        buffer=OUTPUT_MAP_RESOLUTION * 1.5, align_to=align_to, id_prefix=MAP_COUNTRY,
    )
    points = list(zip(grid["utm_x"], grid["utm_y"]))
    patch_ids = list(grid["patch_id"])
    print(f"Sampling grid: {len(grid)} points within the AOI (of {len(grid_full)} in the bounding box)")

In [ ]:
if BUILD_CNN_MAP or BUILD_XGB_MAP:
    ax = boundary_utm.plot(facecolor="none", edgecolor="black", linewidth=2, figsize=(8, 8))
    grid.plot(ax=ax, color="red", markersize=1)
    ax.set_title(f"{MAP_COUNTRY} sampling grid: {len(grid)} points @ {OUTPUT_MAP_RESOLUTION}m")
    plt.show()

## Extract Features

### CNN branch

Samples Sentinel/Landsat patches directly from the COG mosaic at each grid point and runs the 5-fold ResNet50 ensemble.

In [ ]:
if BUILD_CNN_MAP:
    imagery_ds_cls = {"SENTINEL": Sentinel, "LANDSAT": Landsat}[CNN_IMAGERY_TYPE]
    imagery_ds = imagery_ds_cls(
        str(IMAGERY_ROOT),
        filename_glob=f"{MAP_COUNTRY.lower()}_{CNN_IMAGERY_TYPE.lower()}_s*.tif",
        filename_regex=rf"[a-z]{{3}}_{CNN_IMAGERY_TYPE.lower()}_s(?P<date>\d{{4}})",
    )

    cnn_sampler = PointGeoSampler(
        imagery_ds, points_xy=points, points_crs=GRID_CRS, size=PATCH_SIZE, units=Units.PIXELS,
    )
    cnn_loader = DataLoader(
        imagery_ds, sampler=cnn_sampler, batch_size=BATCH_SIZE,
        num_workers=LOADER_WORKERS, collate_fn=stack_samples,
    )
    cnn_transform = ClampAndRescale(mins=SATLAS_MIN_INT.to(DEVICE), maxs=SATLAS_MAX_INT.to(DEVICE))

    model_embeddings_list = []
    model_predictions_list = []

    for split in range(1, 6):

        ckpt = f"{CKPT_STEM}-SPLIT{split}-FINAL.pt"
        model = build_resnet50(hidden_dim=FC_HIDDEN, weights=INITIAL_WEIGHTS)
        embed_dim = model.fc[0].in_features
        model.load_state_dict(torch.load(CHECKPOINT_DIR / ckpt, map_location=DEVICE))
        model.to(DEVICE)

        head = model.fc
        backbone = model
        backbone.fc = torch.nn.Identity()

        features_list = []
        preds_list = []

        head.eval()
        backbone.eval()
        with torch.no_grad():
            for batch in tqdm(cnn_loader, desc=f"  Extracting CNN embeddings for split {split}"):
                x = to_satlas_band_order(batch["image"].float().to(DEVICE))
                x = cnn_transform(x)
                feats = torch.flatten(backbone(x), start_dim=1)
                features_list.append(feats.cpu())
                preds = head(feats)
                preds_list.append(preds.cpu())

        # NOTE: relies on PointGeoSampler (drop_invalid=False, the default) keeping every point,
        # and DataLoader preserving sampler order, so batches line up positionally with patch_ids.
        meta = {
            "patch_id": patch_ids,
            "model_backbone": "resnet50-cnn",
            "model_backbone_name": ckpt,
            "model_backbone_heldoutset": split,
        }
        embed_df = pd.DataFrame(torch.cat(features_list, dim=0).numpy(), columns=[f"X{i}" for i in range(embed_dim)]).assign(**meta)
        model_embeddings_list.append(embed_df)
        pred_df = pd.DataFrame({"wealth_index": torch.cat(preds_list, dim=0).numpy().flatten()}).assign(**meta)
        model_predictions_list.append(pred_df)

    resnet_embeddings = pd.concat(model_embeddings_list).reset_index(drop=True)
    resnet_embeddings.to_parquet(RESNET_EMBEDS_DIR / f"{MAP_COUNTRY.lower()}-embeddings.parquet")

    cnn_predictions = pd.concat(model_predictions_list).reset_index(drop=True)
    cnn_predictions.to_parquet(CNN_PREDICTIONS_OUTDIR / f"{MAP_COUNTRY.lower()}-cnn-predictions.parquet")

    cnn_results = (cnn_predictions
                   .pivot(index="patch_id", columns="model_backbone_heldoutset", values="wealth_index")
                   .rename(columns=lambda f: f"wealth_index_fold{f}")
                   .reset_index())
    cnn_results["wealth_index_mean"] = cnn_results[[f"wealth_index_fold{i}" for i in range(1, 6)]].mean(axis=1)

### Tabular branch

Samples spatial covariates and/or AlphaEarth embeddings at each grid point -- independently, per source, comparing `OUTPUT_MAP_RESOLUTION` against that source's own native COG resolution: a single pixel when they match, or a nodata-masked spatial mean over the point's footprint (requested directly from the torchgeo sampler via `Units.CRS`) otherwise. AEF values are always de-quantised per-pixel before any averaging (see `DequantiseAEF`'s docstring for why order matters).

In [ ]:
if BUILD_XGB_MAP:
    tabular_sources = []
    if covs_cog_path is not None:
        with rasterio.open(covs_cog_path) as src:
            covs_nodata = src.nodata
            covs_native_res = src.transform.a
        covs_ds = SpatialCovariates(
            str(IMAGERY_ROOT),
            filename_glob=f"{MAP_COUNTRY.lower()}_covs_s*.tif",
            filename_regex=r"[a-z]{3}_covs_s(?P<date>\d{4})",
        )
        tabular_sources.append(("covariates", covs_ds, covs_nodata, EXPECTED_COVARIATES, covs_native_res))
    if aef_cog_path is not None:
        with rasterio.open(aef_cog_path) as src:
            aef_native_res = src.transform.a
        aef_bands = [f"A{i:02d}" for i in range(64)]
        aef_ds = AlphaEarth(
            str(IMAGERY_ROOT),
            filename_glob=f"{MAP_COUNTRY.lower()}_aef_s*.tif",
            filename_regex=r"[a-z]{3}_aef_s(?P<date>\d{4})",
        )
        tabular_sources.append(("aef", aef_ds, float("nan"), aef_bands, aef_native_res))

    dequant = DequantiseAEF()
    tabular_features = {}
    for name, ds, nodata, columns, native_res in tabular_sources:

        # Request a single pixel from the torchgeo sampler when OUTPUT_MAP_RESOLUTION matches
        # this source's own native resolution, else a CRS-sized window (averaged below).
        ratio = OUTPUT_MAP_RESOLUTION / native_res
        if math.isclose(ratio, round(ratio), rel_tol=1e-3) and round(ratio) == 1:
            sample_kwargs = dict(size=1, units=Units.PIXELS)
        else:
            sample_kwargs = dict(size=OUTPUT_MAP_RESOLUTION, units=Units.CRS)

        sampler = PointGeoSampler(ds, points_xy=points, points_crs=GRID_CRS, **sample_kwargs)
        loader = DataLoader(
            ds, sampler=sampler, batch_size=TABULAR_BATCH_SIZE,
            num_workers=TABULAR_LOADER_WORKERS, collate_fn=stack_samples,
        )

        key = "image" if ds.is_image else "mask"
        values_list = []
        for batch in tqdm(loader, desc=f"  Sampling {name} for {MAP_COUNTRY}"):
            x = batch[key].float()
            if name == "aef":
                x = dequant(x)
            values_list.append(reduce_spatial_mean(x, nodata=nodata))

        # See the CNN branch's note above -- same positional-order assumption applies here.
        values = torch.cat(values_list, dim=0).numpy()
        df = pd.DataFrame(values, columns=columns)
        df["patch_id"] = patch_ids
        df.to_parquet(TABULAR_FEATURES_DIR / f"{MAP_COUNTRY.lower()}-{name}-{int(OUTPUT_MAP_RESOLUTION)}m.parquet")
        tabular_features[f"cog_{name}"] = df


## Run XGBoost Inference

Filters `XGB_FEATURE_CONFIGS` to whichever entries have all of their required feature sources available this run, merges the corresponding feature frames (all keyed to the same shared grid's `patch_id`), and averages predictions across the 5 spatial-CV folds.

In [ ]:
available_features = {}
if BUILD_XGB_MAP:
    available_features.update(tabular_features)
if BUILD_CNN_MAP:
    available_features["resnet_embeddings"] = resnet_embeddings

xgb_results = {}
for cfg in XGB_FEATURE_CONFIGS:
    feature_type, model_stem, required = cfg["feature_type"], cfg["model_stem"], cfg["requires"]

    missing = [r for r in required if r not in available_features]
    if missing:
        print(f"Skipping {feature_type}: missing feature source(s) {missing}")
        continue

    print(f"Running inference for: {feature_type}")
    predictions_list = []
    for test_fold in tqdm(range(1, 6)):

        trained_model = xgb.XGBRegressor()
        trained_model.load_model(XGB_MODEL_DIR / f"{model_stem}-SPLIT{test_fold}.json")
        feature_names = trained_model.get_booster().feature_names

        frames = []
        for source in required:
            frame = available_features[source]
            if source == "resnet_embeddings":
                frame = frame.query("model_backbone_heldoutset == @test_fold")
            frames.append(frame)

        features = frames[0]
        for frame in frames[1:]:
            features = features.merge(frame, on="patch_id", how="inner")

        X = features[feature_names]
        y = trained_model.predict(X, validate_features=True)
        predictions_list.append(pd.DataFrame({"wealth_index": y, "patch_id": features["patch_id"], "test_fold": test_fold}))

    xgb_output = (pd.concat(predictions_list)
                  .pivot(index="patch_id", columns="test_fold", values="wealth_index")
                  .rename(columns=lambda f: f"wealth_index_fold{f}")
                  .reset_index())
    xgb_output["wealth_index_mean"] = xgb_output[[f"wealth_index_fold{i}" for i in range(1, 6)]].mean(axis=1)
    xgb_output.to_parquet(XGB_PREDICTIONS_OUTDIR / f"{MAP_COUNTRY.lower()}-{feature_type}-xgb-predictions.parquet")
    xgb_results[feature_type] = xgb_output

## Assemble Predictions into Map(s) and Export GeoTIFFs

In [ ]:
if BUILD_CNN_MAP:
    write_wealth_map(
        grid_full, cnn_results, MAPS_DIR / f"{MAP_COUNTRY.lower()}-cnn-wealthmap.tif",
        resolution_m=OUTPUT_MAP_RESOLUTION, epsg_crs=GRID_CRS,
    )

In [ ]:
if BUILD_XGB_MAP:
    for feature_type, results in xgb_results.items():
        write_wealth_map(
            grid_full, results, MAPS_DIR / f"{MAP_COUNTRY.lower()}-{feature_type}-wealthmap.tif",
            resolution_m=OUTPUT_MAP_RESOLUTION, epsg_crs=GRID_CRS,
        )